# 5.7 — Official FastTD3 on MuscleMimic

[![Open In Colab](https://colab.research.google.com/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

One cell: install MyoSuite, then the FastTD3 trainer from `training/fasttd3/` ([younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3)), then smoke-train `myoMimicFullbody-v0`.

- **CPU** (`SMOKE = True`): Gymnasium backend. No Hugging Face token, no Warp.
- **GPU** (`SMOKE = False`): official FastTD3 on the mjlab Warp twin. **Switch the runtime to T4 before setting this.** Runtime → Change runtime type → T4 GPU, then Disconnect and delete runtime, reopen, set `SMOKE = False`.

Keep `SMOKE = True` until the cell prints `ok`. A healthy GPU log starts with `FastTD3 env=... amp=False`. If you see `amp=True` or `buf=` / `upd=`, that is the old homemade TD3 cell.

In [ ]:
from __future__ import annotations

SMOKE = True  # CPU plumbing. False only after Runtime → T4 GPU.
ENV_ID = "myoMimicFullbody-v0"
TRAIN_STEPS = 50_000
N_ENVS = 64  # T4. 1024 OOMs full-body Warp.

import os
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/MyoHub/myosuite.git"
GIT_REF = "tutorial/fasttd3-directional-scratch"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")


def _nvidia_smi() -> subprocess.CompletedProcess[str]:
    return subprocess.run(["nvidia-smi"], capture_output=True, text=True)


def _pip(*pkgs: str) -> None:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        "--constraint",
        str(_KEEP_TORCH),
        *pkgs,
    ]
    subprocess.check_call(cmd)


if not SMOKE:
    smi = _nvidia_smi()
    if smi.returncode != 0:
        raise RuntimeError(
            "SMOKE=False needs an NVIDIA GPU; this session has none "
            f"(nvidia-smi exited {smi.returncode}).\n"
            "Colab: Runtime → Change runtime type → Hardware accelerator = T4 GPU.\n"
            "Then Runtime → Disconnect and delete runtime, reopen this notebook, "
            "set SMOKE=False, and run the cell.\n"
            "On a CPU runtime leave SMOKE=True."
        )
    print(smi.stdout.splitlines()[0] if smi.stdout else "nvidia-smi ok", flush=True)

_KEEP_TORCH = Path("/tmp/keep-torch.txt")
try:
    import torch as _pre_torch

    _KEEP_TORCH.write_text(f"torch=={_pre_torch.__version__.split('+')[0]}\n")
except ImportError:
    _KEEP_TORCH.write_text("")

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )
    dest = Path("/content/myosuite")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(dest), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    else:
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    _pip("-e", f"{dest}[musclemimic]")
    _pip("-e", str(dest / "training" / "fasttd3"))
    repo = dest
else:
    here = Path.cwd().resolve()
    repo = next((d for d in (here, *here.parents) if (d / "myosuite" / "__init__.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Run this notebook from a myosuite clone, or open it in Colab.")
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    pkg = repo / "training" / "fasttd3"
    if str(pkg) not in sys.path:
        sys.path.insert(0, str(pkg))

import torch

print(
    f"IN_COLAB={IN_COLAB} SMOKE={SMOKE} torch={torch.__version__} "
    f"cuda={torch.cuda.is_available()} n={torch.cuda.device_count()}",
    flush=True,
)

out = repo / "runs" / "fasttd3"
cmd = [sys.executable, "-m", "myosuite_fasttd3", ENV_ID, "--out", str(out)]
if SMOKE:
    cmd += ["--smoke", "--backend", "cpu", "--device", "cpu"]
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU runtime is attached but this torch build has no CUDA "
            f"(torch={torch.__version__}). pip likely replaced Colab's GPU wheel.\n"
            "Runtime → Disconnect and delete runtime, pick T4, rerun this cell."
        )
    if IN_COLAB:
        _pip("mjlab>=1.4.0", "mujoco-warp>=3.7.0", "warp-lang>=1.12.0")
    cmd += [
        "--backend",
        "mjlab",
        "--num-envs",
        str(N_ENVS),
        "--total-timesteps",
        str(TRAIN_STEPS),
    ]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd, cwd=str(repo))
print("ok")
